# Deep RecSys Course
## Домашнее задание 4

### ФИО: Хамидуллин Амир 

В этом домашнем задании вы реализуете рекомендательную модель отбора кандидатов на основе архитектуры Transformer. Для этого нужно будет написать датасет, трансформерную архитектуру, саму модель, предсказывающую следующий айтем по истории пользователя, а также трейн и эвал лупы.


### Данные
Данные лежат в архиве `data.zip`, который состоит из:
* `interactions.parquet` - user-item взаимодействия из датасета Yambda (лайки для 500m версии)
* `embeddings.parquet` - уже пофильтрованные и чуть более плотно запакованные эмбеддинги треков из Yambda
* `artists.parquet` - метаданные айтемов с маппингом в артистов

В этом задании нас будет интересовать только файл с взаимодействиями, `interactions.parquet`

Скачать архив можно здесь: [ссылка на google disk](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). В следующем блоке мы скачиваем датасет,поэтому самостоятельно его можно не качать.

### Разбалловка

1. Реализуйте датасеты для обучения и валидации и коллатор для них (2 балла)
1. Реализуйте граф вычислений: трансформер и класс для обучения, выдающий лосс (2 балла)
1. Реализуйте обучение модели (1 балл)
1. Реализуйте валидацию модели (5 баллов)



## Скачивание данных

In [1]:
!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -oq dataset.zip


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
/Users/amirhamidullin/PycharmProjects/recsys/DeepRecSys/.venv/lib/python3.13/site-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=35bbdb18-3475-4f06-bf1e-965291b31aa8
To: /Users/amirhamidullin/PycharmProjects/recsys/DeepRecSys/homeworks/hw4/dataset.zip
100%|████████████████████████████████████████| 356M/356M [02:16<00:00, 2.60MB/s]


## Импорт библиотек

In [2]:
from collections import defaultdict
import gc
import os
from typing import Callable, Dict, List, Tuple, Any, Optional

import numpy as np
import polars as pl

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import tests

In [147]:
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"


# 0. Готовим данные и метрики (копируем с прошлых дз)

Обработка данных

In [ ]:
# Пути к данным (ожидается, что они лежат рядом с ноутбуком)
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# Для воспроизводимости
np.random.seed(42)

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60


data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)


data = data.join(embeddings, on="item_id", how="semi")

temr_df = data['item_id'].value_counts()
valid_df = temr_df.filter(pl.col('count') >= CORE_MIN_INTERACTIONS_PER_ITEM)
data = data.join(valid_df,on ="item_id", how = "semi" )
data = data.join(artists, on = "item_id", how = "left")

# Ремаппинг item_id в непрерывный диапазон [0, N-1].  
unique_items = data["item_id"].unique().sort()
item_map = pl.DataFrame({
    "item_id": unique_items,
    "item_idx": pl.arange(0, len(unique_items), eager=True),
})
data = data.join(item_map, on="item_id").drop("item_id").rename({"item_idx": "item_id"})
print(f"catalog_size: {data['item_id'].n_unique()}")
print(f"max item_id: {data['item_id'].max()}")  # должно быть = n_unique - 1



split_point = data["timestamp"].max() - TEST_INTERVAL_SECONDS
train_df = data.filter(pl.col("timestamp") < split_point)
test_df = data.filter(pl.col("timestamp") >= split_point)

user_relat = train_df['uid'].value_counts()
test_df = test_df.join(user_relat, on="uid", how = "semi")

user_item = test_df.group_by("uid").agg(pl.col('item_id'))
test_targets = dict(user_item.iter_rows())



catalog_size: 157357
max item_id: 157356


Метрики

In [5]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:

    hits = [1 if conditate in set(targets) else 0 for conditate in candidates[:topk]]
    hitrate = int(sum(hits) > 0) # 0 or 1

    recall = sum(hits)/min(len(targets),topk)

    dcg = sum(hits[k-1]/np.log2(k+1) for k in range(1,topk+1))
    idcg = sum(1/np.log2(k+1) for k in range(1, min(topk, len(targets))+ 1 ))
    n_DCG = dcg/idcg if idcg else 0.0

    return {
        "hitrate": hitrate,
        "recall": recall,
        "ndcg": n_DCG,
    }


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    total = {"hitrate": 0.0, "recall": 0.0, "ndcg": 0.0}

    count_users = 0
    for user_id, target in targets.items():
        canditate = candidates[user_id]

        metrics_user = get_metrics(target,canditate, topk)
        for key in total:
            total[key] += metrics_user[key]

        count_users += 1

    total = {key:val/count_users for key,val in total.items()}

    all_canditates_unique = len(set([item for cand in candidates.values() for item in cand[:topk]]))
    # catalog_size_uniq = train.n_unique()
    coverage = all_canditates_unique/catalog_size

    total['coverage'] = coverage

    return total


# 1. Реализуйте датасеты для обучения и валидации и коллатор для них (2 балла)

Аналогично второму домашнему заданию, в этой части вам нужно реализовать вспомогательные функции для работы с пользовательскими историями переменной длины. Эти функции будут использоваться при построении датасета, формировании батчей и подаче данных в модель, поэтому важно заранее зафиксировать формат представления последовательностей. Часть решений из ДЗ 2 можно переиспользовать и здесь.

#### Что концептуально меняется по сравнению с ДЗ 2

Во втором домашнем задании мы из одного пользователя создавали много обучающих примеров (если у него `n` разных покупок — то всего будет `n - 1` обучающих примеров). Каждый пример выглядел как `<таргетное событие, история до этого события>`. Если мы будем обучать какой-то тяжелый энкодер пользователя (e.g., трансформер), нам в рамках обучения надо будет прогнать его `n` раз для почти одного и того же пользователя.

История пользователя: `[e1, e2, e3, e4, e5]`. 
Сэмплы: 
1. `<target: e2, history: [e1]>`
2. `<target: e3, history: [e1, e2]>`
3. `<target: e4, history: [e1, e2, e3]>`
4. `<target: e5, history: [e1, e2, e3, e4]>`

Но если у нас в качестве энкодера используется каузальный трансформер (его еще часто называют декодером) и мы используем в качестве векторного представления пользователя просто скрытое состояние последнего события (или какую-то простенькую агрегацию скрытых состояний из истории), то мы можем сделать гораздо более эффективное обучение в стиле больших языковых моделей. 

А именно — мы можем один раз применить трансформер над последовательностью событий `[e1, e2, e3, e4]`, и получить скрытые состояния пользователя сразу для всех сэмплов — для `history: [e1]`,` history: [e1, e2]`, `history: [e1, e2, e3]`, etc. 

Почему: потому что трансформер каузальный и энкодер при кодировании текущего события (истории на момент текущего события) видит только прошлые события.

Если мы можем за раз получить векторы пользователя для всех сэмплов пользователя, то и обучиться на них сразу можем (посчитать лосс по всем сэмплам пользователя и усреднить).

#### Напоминание: flatten-представление истории

При объединении нескольких пользовательских историй в батч они представляются в flatten-формате: все последовательности конкатенируются в один общий тензор, а их границы задаются отдельным тензором length. Пусть в батч попало несколько пользователей. Тогда их истории объединяются в один “плоский” тензор следующего вида:

`[u1_t1, u1_t2, ..., u1_tL1, u2_t1, ..., u2_tL2, ...]`

В одном тензоре подряд записаны взаимодействия пользователя 1, затем пользователя 2, и так далее.

Чтобы при этом не потерять границы между пользователями, отдельно хранится тензор `length`, где указано, сколько элементов истории относится к каждому пользователю в батче:

`length = [L1, L2, ..., LB]`

где `B` — размер батча, а `Li` — длина истории `i`-го пользователя.

Таким образом:
- на уровне датасета один объект соответствует целой истории пользователя
- на уровне батча несколько таких историй объединяются во flatten-формат
- тензор length позволяет восстановить границы отдельных последовательностей

#### Что нужно реализовать

Ваша задача  реализовать функции, которые позволяют:
- превратить flatten-представление в padded-формат + mask
- реализовать логику создание датасетов для обучения и валидации
- подготовить батч к подаче в модель

### Функция `create_masked_tensor` (возьмите из ДЗ 2)

In [154]:
def create_masked_tensor(
    data: torch.Tensor, lengths: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Converts a batch of flattened variable-length sequences into a padded tensor and mask.
    Supports:
      - indices: data shape (total_num_elements,)
      - embeddings/features: data shape (total_num_elements, d1, d2, ...)

    Parameters
    ----------
    data : torch.Tensor
        Input tensor containing flattened sequences:
        - For indices: shape (total_num_elements,)
        - For embeddings: shape (total_num_elements, embedding_dim)
    lengths : torch.Tensor
        1D tensor of sequence lengths, shape (batch_size,). Specifies the actual length
        of each sequence.

    Returns
    -------
    Tuple[torch.Tensor, torch.Tensor]
        - padded_tensor: Padded tensor of shape:
            - (batch_size, max_seq_len) for indices
            - (batch_size, max_seq_len, embedding_dim) for embeddings
            Shorter sequences are right-padded with zeros.
        - mask: Boolean mask of shape (batch_size, max_seq_len) where True indicates
            valid elements and False indicates padding. Can be used in attention or loss computation.

    Examples
    --------
    >>> data = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
    >>> lengths = torch.tensor([2, 3, 1])
    >>> padded, mask = create_masked_tensor(data, lengths)
    >>> padded
    tensor([[1, 2, 0],
            [3, 4, 5],
            [6, 0, 0]])
    >>> mask
    tensor([[ True,  True, False],
            [ True,  True,  True],
            [ True, False, False]])
    """
    # max_len = max(lengths).item()
    # padded = torch.zeros_like(data.shape)

    # mask = torch.zeros(data.shape, dtype = torch.bool)

    split_history = data.split(lengths.tolist())
    padded = torch.nn.utils.rnn.pad_sequence(split_history, batch_first=True)
    max_len = max(lengths).item()
    mask = torch.arange(max_len, device=lengths.device) < lengths.unsqueeze(-1)

    return padded, mask

In [34]:
tests.test_create_masked_tensor(create_masked_tensor)

All good! :)


### Классы датасетов

Реализуйте классы для обучения и валидации, которые работают с пользовательскими историями взаимодействий и подготавливают семплы.

#### Основная идея

При работе с трансформерами мы задаем максимальную длину последовательности (размер контекста), с которым хотим, чтобы он умел работать. Эта длина задается исходя из ограничений — как по ресурсам (очень большой размер контекста слишком долго обрабатывается или даже не влезает в видеопамять), так и по качеству (иногда на большом контексте модель работает хуже). В простом случае у нас у всех пользователей длины историй меньше заданного ограничения, но на практике встречаются и пользователи с более длинной историей, которых мы за один прогон трансформера обработать не можем. 

И тогда мы делаем чанки: бьём историю пользователя на какие-то сегменты, не превышающие максимальную длину, и каждый из них используем как обучающий пример. В рамках задания вам предлагается просто побить историю пользователя на непересекающиеся сегменты. То есть если история пользователя имеет вид 

`[e1, e2, e3, e4, e5]` и у вас максимальная длина 2, то будут сэмплы 
1. `history: [e1, e2], targets: [e2, e3]`
1. `history: [e2, e3], targets: [e3, e4]`
1. `history: [e3, e4], targets: [e4, e5]`

В этом задании мы не смешиваем логику обучения и валидации в одном классе датасета. Вместо этого нужно реализовать два отдельных датасета:
- `YambdaTrainDataset` — для обучения
- `YambdaEvalDataset` — для валидации

Это делает поведение датасетов более прозрачным: `train`-датасет отвечает только за подготовку обучающих примеров, а `eval`-датасет — только за подготовку пользователей для оценки модели.

#### `YambdaTrainDataset`

Обучающий датасет строится только по `histories`.

Для каждого пользователя его история разбивается на чанки длины не более `max_seq_len`, двигаясь от начала к концу.
Cемпл должен иметь вид:

```python
{
  "uid": uid,
  "history": List[int],
  "targets" : List[int],
  "length": int
}
```

где:
- `history` — входная последовательность айтемов в хронологическом порядке
- `targets` — сдвинутые на один шаг таргеты для `next-item prediction`
- `length` — длина истории (и соответственно таргетов)

#### `YambdaEvalDataset`

Валидационный датасет строится по `histories` и `targets`.

Здесь нужен ровно один семпл на пользователя. В датасет включаются только те пользователи, которые присутствуют в `targets`.

Для каждого такого пользователя нужно взять только хвост его истории длины не более `max_seq_len`.

В отличие от `YambdaTrainDataset` датасета, здесь не нужно разбивать историю на чанки и не нужно строить сдвинутые таргеты.

Cемпл должен иметь вид:

```python
{
  "uid": uid,
  "history": List[int],
  "length": int
}
```

где:
- `history` — хвост истории пользователя в хронологическом порядке
- `length` — длина этой истории

In [41]:
class YambdaTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        self.samples = []

        for u,h in histories.items():
            if len(h) <= 1:
                continue
            
            hist = h[:-1]
            target = h[1:]

            for i in range(0, len(hist), max_seq_len):
                hist_u = hist[i: i + max_seq_len]
                target_u = target[i: i + max_seq_len]

                sample = {"uid":u, "history":hist_u, "targets": target_u, "length": len(hist_u)}
                self.samples.append(sample)
            


    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [42]:
tests.test_yambda_train_dataset(YambdaTrainDataset)

All good! :)


In [45]:
class YambdaEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()
        
        self.samples = []

        for u,h in histories.items():
            if len(h) < 1:
                continue 

            if u in targets:
                h = h[-max_seq_len:]
                sample = {"uid": u, "history": h, "length": len(h)}
                self.samples.append(sample)


    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]


In [46]:
tests.test_yambda_eval_dataset(YambdaEvalDataset)

All good! :)


### Функция `collate_fn`

Реализуйте функцию `collate_fn`, которая будет использоваться в `DataLoader` для преобразования списка семплов
из `YambdaTrainDataset` и `YambdaEvalDataset` в батчи для подачи в модель и работы с ними.

Как говорилось ранее, мы используем flatten-представление:
- конкатенируем все пользовательские истории в батче в один 1D-тензор  
- отдельно сохраняем `length`, чтобы позже восстановить границы последовательностей

#### Вход

`batch: List[Dict[str, Any]]` — список семплов из `YambdaTrainDataset` или `YambdaEvalDataset`.

#### Что должна делать `collate_fn`

Функция должна как и раньше:
- Cформировать единый словарь, где все значения — `torch.Tensor` типа `torch.long`.
- Не использовать `padding`. Только `flatten`-конкатенация + `lengths`.
- Сохранять порядок объектов в `batch` при конкатенации.
- Все числовые значения должны быть приведены к `torch.Tensor` типа `torch.long`.


In [51]:

def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    is_train = False
    is_eval = False

    if batch[0].get("targets"): is_train = True
    else: is_eval = True

    res = {"uid": [], "history": [], "length": []}
    if is_train:
        res["targets"] = []

    for b in batch:
        res["uid"].append(b["uid"])
        res["history"].extend(b["history"])

        if is_train:
            res["targets"].extend(b["targets"])
        
        res["length"].append(b["length"])

    
    res["uid"] = torch.tensor(res["uid"], dtype = torch.long)
    res["history"] = torch.tensor(res["history"], dtype = torch.long)
    if is_train:
        res["targets"] = torch.tensor(res["targets"], dtype = torch.long)
    res["length"] = torch.tensor(res["length"], dtype = torch.long)


    return res

In [52]:
tests.test_collate_fn(collate_fn)

All good! :)


Посмотрим на результат

Что нужно задать в переменных:

- `data` — уже предобработанный датафрейм взаимодействий (`uid`, `item_id`, `timestamp`).
- `train_df` — строки из `data` с `timestamp < test_start_ts`.
- `test_df` — строки из `data` с `timestamp >= test_start_ts`.
- `catalog_size` — число уникальных `item_id` в `data`.

Важно: здесь не нужно повторно делать remap `item_id` через `item_mapping`, если он уже был сделан выше в ноутбуке.

In [55]:
split_point = data["timestamp"].max() - TEST_INTERVAL_SECONDS
train_df = data.filter(pl.col("timestamp") < split_point)
test_df = data.filter(pl.col("timestamp") >= split_point)
catalog_size = data["item_id"].n_unique()


In [56]:
tests.test_split_fields(
    data=data,
    train_df=train_df,
    test_df=test_df,
    catalog_size=catalog_size,
)

All good! :)


In [57]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
CATALOG_SIZE = data["item_id"].n_unique()

histories = {
    row["uid"]: list(row["item_id"])
    for row in train_df.group_by("uid", maintain_order=True)
    .agg(pl.col("item_id").sort_by("timestamp"))
    .iter_rows(named=True)
}

raw_targets = dict(test_df.group_by("uid").agg(pl.col("item_id")).iter_rows())
targets = {
    uid: t
    for uid, t in raw_targets.items()
    if uid in histories and len(histories[uid]) > 0
}


yambda_train_dataset = YambdaTrainDataset(histories=histories)
yambda_eval_dataset = YambdaEvalDataset(histories=histories, targets=targets)

yambda_train_dataloader = DataLoader(
    dataset=yambda_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True,
)

yambda_eval_dataloader = DataLoader(
    dataset=yambda_eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False,
)

In [58]:
tests.test_dataloaders(
    train_dataloader=yambda_train_dataloader,
    eval_dataloader=yambda_eval_dataloader,
)

All good! :)


In [59]:
train_batch = next(iter(yambda_train_dataloader))
eval_batch = next(iter(yambda_eval_dataloader))

print("TRAIN BATCH")
print({k: tuple(v.shape) for k, v in train_batch.items()})
print("uid[:5]:", train_batch["uid"][:5].tolist())
print("length[:5]:", train_batch["length"][:5].tolist())
print("history[:20]:", train_batch["history"][:20].tolist())
print("targets[:20]:", train_batch["targets"][:20].tolist())

print("\nEVAL BATCH")
print({k: tuple(v.shape) for k, v in eval_batch.items()})
print("uid[:5]:", eval_batch["uid"][:5].tolist())
print("length[:5]:", eval_batch["length"][:5].tolist())
print("history[:20]:", eval_batch["history"][:20].tolist())

TRAIN BATCH
{'uid': (128,), 'history': (8623,), 'length': (128,), 'targets': (8623,)}
uid[:5]: [490070, 798280, 988310, 744120, 179720]
length[:5]: [100, 100, 100, 100, 5]
history[:20]: [19741, 92972, 145338, 53680, 37302, 37157, 88096, 101069, 95453, 129332, 19385, 22224, 130430, 105085, 37190, 72477, 11399, 60314, 55705, 29073]
targets[:20]: [92972, 145338, 53680, 37302, 37157, 88096, 101069, 95453, 129332, 19385, 22224, 130430, 105085, 37190, 72477, 11399, 60314, 55705, 29073, 26066]

EVAL BATCH
{'uid': (128,), 'history': (8876,), 'length': (128,)}
uid[:5]: [30, 40, 50, 110, 130]
length[:5]: [100, 100, 95, 100, 19]
history[:20]: [100048, 64614, 88110, 40419, 30050, 141173, 4001, 74098, 95289, 3813, 75911, 29932, 139871, 112280, 134886, 119362, 55818, 47279, 34283, 51364]


# 2. Реализуйте граф вычислений: трансформер и класс для обучения, выдающий лосс (2 балла)

В этом домашнем задании в качестве функции агрегации последовательности мы будем использовать "GPT-style Transformer". В отличие от двунаправленного `self-attention`, здесь применяется каузальная маска: представление объекта на позиции `t` может зависеть только от объектов на позициях `0, 1, ..., t`, но не от будущих элементов последовательности.

Такой подход хорошо подходит для задач последовательного моделирования, где модель должна предсказывать следующий элемент только по истории пользователя.

В этой части домашнего задания вам нужно реализовать основные компоненты такого Transformer'а:
- каузальный Multi-Head Self-Attention
- position-wise MLP
- Transformer block
- комбинацию из нескольких блоков
- финальную модель, которая применяет трансформер для получения пользовательского представления и вычисляет лосс



## Реализация слоя внимания с каузальной маской

Реализуйте класс `CausalSelfAttention`, который выполняет `multi-head self-attention` с каузальной маской.

На вход модуль получает:
- тензор `x` формы `(B, S, D)`, где
    - `B` — размер батча (`batch_size`)
    - `S` — длина последовательности (`seq_len`)
    - `D` — размер скрытого представления (`embedding_dim`)
- булеву маску `mask` формы `(B, S)`, где `True` означает валидную позицию, а `False` паддинг.

#### Необходимо:
- спроецировать вход в `Q, K, V` тензора
- разбить их на `n_heads` голов
- построить attention mask, которая:
    - запрещает смотреть в будущее
    - запрещает смотреть на позиции с паддингом
- применить `scaled_dot_product_attention` с каузальной маской
- объединить головы обратно
- применить финальную линейную проекцию

#### Зачем это нужно

Это основной механизм transformer-модели, который позволяет каждому элементу последовательности агрегировать информацию из предыдущего контекста. Каузальная маска гарантирует, что на позиции `t` модель использует только элементы с позиций `<= t`, а padding-маска исключает влияние фиктивных токенов.

In [79]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads

        self.h_dim = d_model // n_heads

        self.Wqkv = nn.Linear(d_model, d_model * 3)
        self.Wo = nn.Linear(d_model, d_model)
        self.dropout_p = dropout   
        

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B,L,D = x.shape
        KQV = self.Wqkv(x) # (b,l,d) -> b,l,d_model * 3
        Q,K,V = torch.chunk(KQV, 3, dim = -1) # b, l, d for all

        Q = Q.view(B,L,self.n_heads, self.h_dim).transpose(1, 2) # b, n_heads, l, h_dim
        K = K.view(B,L,self.n_heads, self.h_dim).transpose(1, 2)
        V = V.view(B,L,self.n_heads, self.h_dim).transpose(1, 2)

        causal_mask = torch.ones(L, L, device=x.device, dtype=torch.bool).tril()
        combined_mask = mask.unsqueeze(1).unsqueeze(2) & causal_mask
        
        out = F.scaled_dot_product_attention(Q, K, V, attn_mask=combined_mask, dropout_p=self.dropout_p if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, L, D)  # (B, n_heads, L, h_dim) -> (B, L, D)

        return self.Wo(out)



In [76]:
a = nn.Linear(3,3*4)
# print(a.weight)

x = torch.rand(3,1).T
print(x)
c = a(x)
# print(c)

num_h = 3
h_dim = 8
print()

s = torch.rand(2,3,6)
# print(s)

# print(s.view(4,3,2))
# print(s.view(4,3,4))
# print(s.view(4,3,4,3))
# print(s.view(2,3,num_h, h_dim))

x = torch.rand(3,4,5)
print(x.T.shape)



tensor([[0.8347, 0.2539, 0.6015]])

torch.Size([5, 4, 3])


In [80]:
tests.test_causal_self_attention(CausalSelfAttention)

All good! :)


## Реализация MLP части

Реализуйте двухслойный `position-wise MLP`, который применяется независимо к каждой позиции в последовательности.

На вход модуль получает тензор `x` формы `(B, S, D)` и возвращает тензор той же формы, где:
- `B` — размер батча
- `S` — длина последовательности
- `D` — размер скрытого представления

Типичная структура MLP:
1. линейная проекция `d_model -> 4 * d_model`
1. нелинейность `GELU`
1. `dropout`
1. линейная проекция `4 * d_model -> d_model`

#### Зачем это нужно

Модуль `self-attention` смешивает информацию между позициями последовательности, а `MLP` выполняет нелинейное преобразование внутри каждой позиции отдельно. Вместе эти два компонента образуют стандартный `Transformer block`.

In [84]:
class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()


        self.linear1 = nn.Linear(d_model, 4*d_model)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(4*d_model,d_model)
        self.drop = nn.Dropout(dropout)


        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res =  self.linear2(self.drop(self.activation(self.linear1(x))))
        return res


In [85]:
tests.test_mlp(MLP)

All good! :)


## Реализация блока трансформера

Реализуйте один `Transformer block` в `Pre-LayerNorm` постановке.

`Transformer block` должен состоять из двух последовательных residual блоков:
- `LayerNorm -> CausalSelfAttention -> Dropout -> Residual`
- `LayerNorm -> MLP -> Dropout -> Residual`

То есть вход `x` должен обрабатываться сначала через  `CausalSelfAttention` блок, затем через `MLP` блок.

#### Зачем это нужно

Трансформер строится как последовательное применение одинаковых блоков. Каждый такой блок постепенно улучшает представления элементов последовательности, комбинируя контекстное взаимодействие через механизм внимания и нелинейную обработку через `MLP`.

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.mlp = MLP(d_model, dropout)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout) 

        
        

    def forward(self, x, mask=None):
        if mask is None:
            mask = torch.ones(x.shape[0], x.shape[1], dtype=torch.bool, device=x.device)
    
        h = x + self.drop1(self.attn(self.ln1(x), mask))       
        out = h + self.drop2(self.mlp(self.ln2(h)))            
        return out



In [87]:
tests.test_block(Block)

All good! :)


## Реализация энкодера 

Реализуйте "GPT-style" энкодер, который принимает на вход уже построенные представления последовательности и последовательно обрабатывает их стеком transformer-блоков.

#### Модель должна:
1. принять входной тензор `x` формы `(B, S, D)`
1. применить `dropout` к входным представлениям
1. последовательно пропустить результат через `n_layers` блоков
1. применить финальный `LayerNorm`

На выходе модель должна возвращать скрытые состояния формы `(B, S, D)`.

#### Зачем это нужно

Это основная последовательная модель, которая строит контекстные представления для всех позиций последовательности с учетом каузального ограничения. Далее эти представления можно использовать для предсказания следующего элемента последовательности.

In [ ]:
class GPT(nn.Module):
    def __init__(
        self,
        max_seq_len: int,
        n_layers: int,
        d_model: int,
        n_heads: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.max_seq_len =max_seq_len
        self.drop = nn.Dropout(dropout)
        

        list_l = []
        for l in range(n_layers):
            self.layer = Block(d_model, n_heads, dropout)
            list_l.append(self.layer)
        
        self.layers = nn.ModuleList(list_l)
        self.layernorm = nn.LayerNorm(d_model)
        

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        if mask is None:
            mask = torch.ones(x.shape[0], x.shape[1], dtype=torch.bool, device=x.device)


        assert x.shape[1] <= self.max_seq_len
        res = self.drop(x)

        for l in self.layers:
            res = l(res, mask)
        
        res = self.layernorm(res)
        return res

In [ ]:
# x = torch.rand((3,4,5))
# print(nn.Dropout(0.1)(x))

tensor([[[0.0000, 0.0000, 0.0050, 0.4556, 0.0510],
         [0.3695, 0.7242, 0.9483, 0.8650, 0.0000],
         [0.7860, 0.2110, 0.1144, 0.5786, 1.0831],
         [0.5868, 0.5078, 0.1785, 0.2801, 0.1273]],

        [[1.0004, 0.0000, 0.0648, 0.6152, 0.5959],
         [0.4869, 0.7544, 0.8543, 0.0331, 0.1299],
         [0.5289, 0.0000, 0.0509, 1.0461, 0.5133],
         [0.2809, 0.3650, 0.1218, 0.9891, 0.4351]],

        [[0.2432, 0.0000, 0.8803, 0.4788, 0.4972],
         [0.4108, 0.2874, 0.6544, 0.9830, 0.2336],
         [0.5822, 0.1330, 0.2951, 1.0432, 0.5614],
         [1.0907, 0.4582, 0.2528, 0.0000, 0.0000]]])


In [99]:
tests.test_gpt(GPT)

All good! :)


## UserEncoder

Реализуйте класс `UserEncoder`, который строит контекстные представления пользователя в каждый момент его истории на основе `GPT-style Transformer`.

На вход модуль получает батч пользовательских историй в flatten-формате:

#### Что должна делать модель

1. Преобразовать `history` из айдишников айтемов в эмбеддинги через lookup слой (`nn.Embedding`)
1. По `length` восстановить границы пользовательских последовательностей и перейти к padded-представлению
1. Добавить позиционные эмбеддинги
1. Подать полученные последовательности в `GPT` энкодер 
1. Вернуть контекстные представления всех пользователей в flatten-формате

Если суммарное число событий в батче равно `N`, а размерность представлений равна `D`, то выход должен иметь форму `(N, D)` (аналогичную входу).

#### Зачем это нужно

`UserEncoder` преобразует историю взаимодействий в набор контекстных представлений, где каждое состояние соответствует некоторой позиции последовательности и учитывает все предыдущие события. Далее эти представления можно использовать для обучения модели на задачу `next item prediction`.


In [ ]:
class UserEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.max_seq_len = max_seq_len

        self.item_embeddings = nn.Embedding(num_items, embedding_dim)

        self.positional = nn.Embedding(max_seq_len, embedding_dim)
        self.gpt = GPT(max_seq_len,n_layers,embedding_dim, n_heads, dropout)



    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:

        item_to_embed = self.item_embeddings(inputs["history"]) # lookup
        padded, mask = create_masked_tensor(item_to_embed, inputs["length"])

        positions = torch.arange(padded.shape[1], device=padded.device)
        positions_emb = self.positional(positions)

        final_emb = (padded + positions_emb) * mask.unsqueeze(-1)
        res = self.gpt(final_emb, mask)

        lengths = inputs["length"]

        # filtred_res = []
        
        # for i in range(len(lengths)):
        #     filtred_res.append(res[i, :lengths[i]])

        res = res[mask]
        
        
        


        return res
        



In [166]:
gc.collect()
torch.cuda.empty_cache()

train_graph = TrainNIPModel(
    num_items=catalog_size,
    embedding_dim=64,
    num_negatives=512,
    q_counts=q,
).to(DEVICE)
optimizer = torch.optim.Adam(params=train_graph.parameters(), lr=LEARNING_RATE)

_, epoch_losses = train(
    dataloader=yambda_train_dataloader,
    model=train_graph,
    optimizer=optimizer,
    num_epochs=1,
)

tests.test_train_loop(epoch_losses[0])

  2%|▏         | 15/981 [01:28<1:35:10,  5.91s/it]


KeyboardInterrupt: 

In [110]:
# inputs = {
#     "history": torch.tensor([10, 20, 30, 40, 50]),   # (N,) — flat item_ids
#     "length":  torch.tensor([3, 2]),                  # (B,) — длины
#     "uid":     torch.tensor([0, 1]),                  # (B,) — id пользователей
#     # "targets": tensor([...])                  # (N,) — только при обучении
# }
# E = nn.Embedding(100, 4)

# E(inputs["history"])


hidden_st = torch.rand(2,4,3) # B L D
print(hidden_st)
print('--------')
l = torch.tensor([2,3]) # B,


for i in range(len(l)):
    print(hidden_st[i, :l[i]])
    print('---')





tensor([[[0.4219, 0.4458, 0.0919],
         [0.0399, 0.1005, 0.1439],
         [0.8781, 0.1477, 0.3190],
         [0.4281, 0.5533, 0.5117]],

        [[0.6921, 0.2281, 0.8506],
         [0.9556, 0.2712, 0.4233],
         [0.0841, 0.1662, 0.2208],
         [0.2707, 0.6050, 0.7176]]])
--------
tensor([[0.4219, 0.4458, 0.0919],
        [0.0399, 0.1005, 0.1439]])
---
tensor([[0.6921, 0.2281, 0.8506],
        [0.9556, 0.2712, 0.4233],
        [0.0841, 0.1662, 0.2208]])
---


In [159]:
tests.test_user_encoder(UserEncoder)

All good! :)


## TrainNIPModel

Реализуйте класс `TrainNIPModel`, который объединяет `UserEncoder` с логикой обучения модели на задачу `next-item prediction`.

#### Общая идея

В этом задании мы с вами построим полноценную двухбашенную модель. Пользовательская башня — это `UserEncoder`. Она получает последовательность событий из истории пользователя и строит скрытое представление каждого префикса истории. Айтемная башня — это простая таблица эмбеддингов объектов (`nn.Embedding`). То есть для каждого айтема мы получаем его представление через обычный `lookup` из этой матрицы.

Важно, что эта это матрица используется сразу в двух башнях:
- как представление событий на входе в `UserEncoder`
- как представление целевых и негативных айтемов при вычислении лосса

Иными словами, отдельного сложного айтемного энкодера здесь нет: в качестве айтемной башни выступает один эмбединг-слой

#### Что получает модель на вход

На вход модель получает батч `inputs`, содержащий:
- `history` — последовательности айтемов из пользовательских историй
- `targets` — сдвинутые таргеты для обучения

`history` используется в `UserEncoder`, который возвращает скрытые состояния для всех позиций всех историй в батче.
`targets` используется для подсчета лосса.

#### Функция `forward`

Функция должна:
1. Передать `inputs` через `UserEncoder`
1. Получить скрытые состояния всех позиций истории
1. Передать их вместе с `inputs` в `compute_loss`
1. Вернуть итоговый `loss`

#### Как устроено обучение 

Мы хотим обучать нашу модель на задачу `next item prediction`. 

Пусть история пользователя имеет вид:

$$(i_1, i_2, \cdots, i_L),$$

Тогда скрытое состояние на позиции $t$ должно использоваться для предсказания следующего объекта, то есть $i_{t+1}$.

Однако в этом задании train-датасет уже заранее формирует пары вида:
- `history` = $[i_1, i_2, \dots, i_{L-1}]$
- `targets` = $[i_2, i_3, \dots, i_L]$

Поэтому модель обучается авторегрессионно сразу по всем позициям последовательности:
- по представлению для $i_1$ предсказывается $i_2$
- по представлению для $i_2$ предсказывается $i_3$
- ...
- по представлению для $i_{L-1}$ предсказывается $i_L$

Таким образом, `encoder_output` содержит представления пользовательских префиксов, а `targets` уже содержит позитивы для каждой позиции.

#### Функция compute_loss

В `compute_loss` нужно реализовать sampled softmax с in-batch негативами и оригинальной (не fixed!) logq-коррекцией, как во втором домашнем задании.
Идея:
- `encoder_output` используется как представление пользователя для каждой позиции
- позитив для каждой позиции берется из `targets`
- негативы формируются из других `target`-айтемов в текущем батче
- к логитам применяется logq-коррекция

#### LogQ-correction

При использовании in-batch негативов разные объекты появляются в батче с разной частотой. Из-за этого распределение негативов получается неравномерным. Чтобы скорректировать этот сдвиг, нужно применить logq-коррекцию как это делалось во втором домашнем задании.

Функция `compute_loss` должна:
1. Взять скрытые представления из `encoder_output` и соответствующие метки из `targets`
1. Получить эмбеддинги этих для `targets` через матрицу эмбедингов айтемов
1. Cформировать матрицу логитов между пользовательскими представлениями и in-batch кандидатами
1. Применить logq-коррекцию к логитам кандидатов
1. Посчитать итоговый `loss`

Как итог, модель учится строить представление каждого префикса истории пользователя так, чтобы оно было близко к представлению его следующего взаимодействия и отделяло его от прочих айтемов из каталога.

### Подготовка статистик для logq-коррекции

В `TrainNIPModel` используется `logq`-коррекция для in-batch негативов. В этой части ДЗ нужно подготовить **сырые количества** по train-targets, а не заранее вычисленные частоты.

Почему так:

- мы не хотим хранить и передавать отдельный вектор частот `q`;
- для численной стабильности удобнее работать в лог-пространстве;
- `logq` можно вычислять прямо в `forward/compute_loss` через counts:
  - `logq(i) = log(count(i)) - log(sum_j count(j))`.

Что нужно реализовать:

- собрать все `target`-айтемы из train-датасета;
- посчитать для каждого `item_id` количество появлений в train-targets;
- вернуть тензор длины `catalog_size`, где `q_counts[i] = count(item_i)`.

Важно:

- считаем статистику именно по `targets`, а не по `history`;
- здесь не нормализуем counts в вероятности;
- нормализация и логарифмы делаются в лоссе во время `forward`.

In [155]:
def build_q_from_train_targets(
    train_targets: torch.Tensor,
    catalog_size: int,
) -> torch.Tensor:

    if len(train_targets) == 0:
        raise ValueError("empty")
    if train_targets.max() >= catalog_size or train_targets.min() < 0:
        raise ValueError("invalid ids")




    cnts = torch.bincount(train_targets.flatten(),minlength=catalog_size).float()
    return cnts


train_target_ids = torch.tensor(
    [
        target
        for idx in range(len(yambda_train_dataset))
        for target in yambda_train_dataset[idx]["targets"]
    ],
    dtype=torch.long,
)


q = build_q_from_train_targets(
    train_targets=train_target_ids,
    catalog_size=catalog_size,
)

In [134]:
tests.test_logq_coefficients(build_q_from_train_targets)

All good! :)


In [141]:
class TrainNIPModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.num_negatives = num_negatives
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer("q_counts", q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if "weight" in key:
                nn.init.trunc_normal_(
                    value.data,
                    std=initializer_range,
                    a=-2 * initializer_range,
                    b=2 * initializer_range,
                )
            elif "bias" in key:
                nn.init.zeros_(value.data)

    def compute_loss(self, encoder_output, inputs):
        targets = inputs["targets"]
        N = targets.shape[0]

        neg_pos = torch.randint(0, N, (N, self.num_negatives), device=targets.device)
        neg_ids = targets[neg_pos]                                   

        candidate_ids = torch.cat([targets[:, None], neg_ids], dim=1) 
        candidate_emb = self.encoder.item_embeddings(candidate_ids)   
        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1) 

        q = self.q_counts[neg_ids]                                   
        logq = torch.log(q + self.eps) - torch.log(self.q_counts.sum() + self.eps)
        logits[:, 1:] = logits[:, 1:] - logq

        labels = torch.zeros(N, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        return self.compute_loss(encoder_output, inputs)



In [142]:
tests.test_train_nip_model(TrainNIPModel)

All good! :)


# 3. Реализуйте обучение модели (1 балл)

Реализуйте функцию `train`, которая обучает модель. В этот раз мы пока что не будем запускать валидацию во время обучения, поэтому трейн-луп включает только проход по всем батчам train-датасета и шаги оптимизации.

Функция должна вернуть **две отдельные переменные**:
- `checkpoint`: состояние модели (`state_dict`) после обучения;
- `epoch_losses`: список средних train-loss по эпохам (для контроля качества обучения).

Ожидаемый формат возврата: `return checkpoint, epoch_losses`.

In [163]:
from tqdm import tqdm
def train(
    dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    device: str = "mps",
) -> tuple[dict[str, Any], list[float]]:

    model.to(device)
    avg_loss_per_e = []
    model.train()
    

    for e in range(num_epochs):
        avg_l = 0
        cnt_b = 0
        for b in tqdm(dataloader):
            b = {k: v.to(device) for k, v in b.items()}
            
            loss = model(b)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            avg_l += loss.item()
            cnt_b +=1 
        
        avg_l = avg_l / cnt_b
        avg_loss_per_e.append(avg_l)


    print(avg_loss_per_e)

    return (model.state_dict(), avg_loss_per_e)

In [164]:
gc.collect()
torch.cuda.empty_cache()

train_graph = TrainNIPModel(
    num_items=catalog_size,
    embedding_dim=64,
    num_negatives=512,
    q_counts=q,
).to(DEVICE)
optimizer = torch.optim.Adam(params=train_graph.parameters(), lr=LEARNING_RATE)

_, epoch_losses = train(
    dataloader=yambda_train_dataloader,
    model=train_graph,
    optimizer=optimizer,
    num_epochs=1,
)

tests.test_train_loop(epoch_losses[0])

Python(9637) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
  2%|▏         | 20/981 [02:18<1:51:10,  6.94s/it]


KeyboardInterrupt: 

# 4. Реализуйте валидацию модели (5 баллов)

## EvalNIPModel

Реализуйте класс `EvalNIPModel`, который используется на этапе инференса и оценки модели.

#### Общая идея

Во время обучения модель строила представление каждого префикса пользовательской истории, чтобы по нему предсказывать следующий айтем. На этапе инференса нам обычно не нужны представления всех позиций сразу. Вместо этого нужно получить одно итоговое представление пользователя — по его последнему доступному событию в истории.

#### Что получает модель на вход

На вход модель получает батч `inputs`, содержащий `history`. `history` описывает последовательности событий пользователей. После пропуска через `UserEncoder` мы получаем скрытые состояния для всех позиций всех историй в батче.

#### Функция forward
Функция должна:
- Передать `inputs` в `UserEncoder`
- Получить скрытые состояния всех позиций истории
- Для каждого пользователя выбрать скрытое состояние, соответствующее последнему событию в его истории
- Посчитать скоры релевантности между каждым пользователем и каждым объектом каталога
- Вернуть матрицу `scores` размера `[B, num_items]`, где:
    - `B` — число пользователей в батче
    - `num_items` — число айтемов в каталоге


#### Как считаются скоры

Релевантность считается как скалярное произведение между финальным пользовательским представлением и эмбеддингом каждого айтема из каталога. Для каждого пользователя мы получаем скор для каждого объекта.


In [ ]:
class EvalNIPModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

In [ ]:
tests.test_eval_nip_model(EvalNIPModel)

## Функция `eval` 

Реализуйте функцию `eval`, которая проходит по всему датасету и оценивает качество рекомендательной модели.

#### Общая идея

На этапе оценки модель должна для каждого пользователя:
1. Построить скоры релевантности по всему каталогу
1. Выбрать top-k наиболее релевантных объектов
1. Сохранить эти рекомендации
1. После прохода по всему датасету посчитать итоговые метрики качества.

#### Что делает функция
Функция должна:
- Перевести модель в режим `eval`
- Пройти по всем батчам из `dataloader`
- Для каждого батча получить матрицу скоров размера `[B, num_items]`
- Для каждого пользователя выбрать top-k айтемов с наибольшими скорами
- Собрать все предсказания в словарь формата `Dict[uid, List[item_id]]`
- После обработки всего датасета вызвать переданную функцию `evaluate_fn(...)` с ground truth `targets` и вернуть словарь метрик

#### Tips & Tricks
- Не забудьте перевести модель в режим `model.eval()`
- На этапе оценки градиенты не нужны, поэтому используйте `torch.no_grad()` / `torch.inference_mode()`
- После получения скоров по каталогу нужно взять top-k лучших объектов через `torch.topk`
- Метрики считаются и усредняются не по отдельному батчу, а по всем пользователям из датасета
- Сигнатура принимает `targets` (разметка для оценки) и `evaluate_fn` (например ваша `evaluate` из предыдущей части ДЗ), чтобы не полагаться на глобальные переменные

In [ ]:
def eval(
    dataloader: DataLoader,
    model: torch.nn.Module,
    catalog_size: int,
    topk: int,
    device: str = "cuda",
    *,
    targets: Dict[int, List[int]],
    evaluate_fn: Callable[..., Dict[str, float]],
) -> Dict[str, float]:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
tests.test_eval(eval)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

train_graph = TrainNIPModel(
    num_items=catalog_size,
    embedding_dim=64,
    num_negatives=512,
    q_counts=q,
).to(DEVICE)
optimizer = torch.optim.Adam(params=train_graph.parameters(), lr=LEARNING_RATE)

checkpoint, epoch_losses = train(
    dataloader=yambda_train_dataloader,
    model=train_graph,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
)

test_dataset = yambda_eval_dataset
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False,
)

test_graph = EvalNIPModel(
    num_items=catalog_size,
    embedding_dim=64,
).to(DEVICE)
encoder_state = {
    key.removeprefix("encoder."): value
    for key, value in checkpoint.items()
    if key.startswith("encoder.")
}
test_graph.encoder.load_state_dict(encoder_state)

final_metrics_nip = eval(
    dataloader=test_dataloader,
    model=test_graph,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE,
    targets=targets,
    evaluate_fn=evaluate,
)
print(final_metrics_nip)
tests.check_nip_recs(final_metrics_nip)